# Building Iterative Multi-Agent Systems
## From Task → Evaluation → Continuous Improvement Loops → Multi-Agent Orchestration

**A practical, hands-on course** (November 2026)

This notebook walks you through the full progression:
1. Defining a clear, evaluable task
2. Evaluating an agent on that task
3. Turning evaluation into a **bounded iterative improvement loop**
4. Orchestrating multiple specialized agents around that loop

You will implement the core **Generator-Critic** pattern and see how it maps to graph-based orchestration (LangGraph style).

## Module 1: Creating a Clear Task for an Agent

A good task has:
- **Goal** — what success looks like
- **Success Criteria** — explicit checklist / rubric
- **Constraints** — format, length, tools, safety
- **Output Schema** — preferred structure

### Example Task

In [ ]:
TASK = """
Task: Write a concise technical summary of a paper.

Goal: Produce a 300-500 word summary that a senior engineer can use to decide whether to read the full paper.

Success Criteria:
- Accurately captures the problem, method, key results, and limitations
- Includes quantitative findings where present
- Flags any claims that lack strong evidence
- Ends with 2-3 concrete implications or open questions
- Uses clear language

Constraints:
- Stay under 500 words
- Do not invent numbers or citations
- Output in Markdown with headings: Problem, Method, Results, Limitations, Implications
"""

print("Task defined.")
print(TASK[:400] + "...")

## Module 2: Evaluating the Agent

We use an **LLM-as-Judge** (or a simple scoring function for demo) that returns structured feedback.

In [ ]:
import json
import random

def mock_evaluator(task: str, output: str) -> dict:
    """Mock evaluator for demonstration (replace with real LLM call)."""
    # Simple heuristic demo: longer + more headings = higher score
    score = 5.0
    issues = []
    
    if len(output) < 200:
        score -= 2
        issues.append("Too short")
    if "Problem" not in output or "Results" not in output:
        score -= 1.5
        issues.append("Missing required sections")
    if "implications" not in output.lower() and "Implication" not in output:
        score -= 1
        issues.append("Missing implications")
    
    score = max(1.0, min(10.0, score + random.uniform(-0.5, 1.0)))
    
    return {
        "score": round(score, 1),
        "passed": score >= 8.0,
        "issues": issues,
        "feedback": "; ".join(issues) if issues else "Looks solid. Minor polish only."
    }

# Test the evaluator
sample = "Problem: ... Method: ... Results: numbers... Limitations: ... Implications: 1. 2."
print(json.dumps(mock_evaluator(TASK, sample), indent=2))

## Module 3: The Iterative Improvement Loop (Generator-Critic)

Core pattern:
1. Generator produces output
2. Evaluator scores + gives feedback
3. If not good enough and under max attempts → revise with feedback
4. Hard stop on max iterations or stagnation

In [ ]:
def mock_generator(task: str, feedback: str = "") -> str:
    """Mock generator that improves when feedback is given."""
    base = """# Technical Summary

## Problem
The paper addresses scalability issues in multi-agent systems.

## Method
They propose a cyclic graph with conditional routing.

## Results
Accuracy improved by 12% on the benchmark.

## Limitations
Limited to English and small model sizes.
"""
    if feedback:
        base += f"\n## Implications\n1. Apply the same pattern to production workflows.\n2. Add stagnation detection.\n(Revised based on: {feedback[:60]}...)"
    else:
        base += "\n(Initial draft — missing implications)"
    return base

def iterative_loop(task: str, max_attempts: int = 3, threshold: float = 8.0):
    feedback = ""
    history = []
    best = None
    
    for attempt in range(1, max_attempts + 1):
        print(f"\n=== Attempt {attempt}/{max_attempts} ===")
        output = mock_generator(task, feedback)
        evaluation = mock_evaluator(task, output)
        
        history.append({
            "attempt": attempt,
            "score": evaluation["score"],
            "feedback": evaluation["feedback"],
            "output_preview": output[:200] + "..."
        })
        
        print(f"Score: {evaluation['score']} | Passed: {evaluation['passed']}")
        print(f"Feedback: {evaluation['feedback']}")
        
        if best is None or evaluation["score"] > best["score"]:
            best = {"score": evaluation["score"], "output": output}
        
        if evaluation["passed"]:
            print("✅ Threshold met. Stopping.")
            return output, history, best
        
        feedback = evaluation["feedback"]
    
    print("⚠️ Max attempts reached. Returning best version.")
    return best["output"], history, best

final_output, history, best = iterative_loop(TASK)
print("\n--- Final Output (preview) ---")
print(final_output[:500])

## Module 4: Multi-Agent Orchestration

Expand the single loop into specialized agents:
- **Researcher** — gathers facts
- **Writer** — drafts from research
- **Critic** — evaluates against rubric
- **Orchestrator** — routes (revise writer / request more research / accept)

In production this is best expressed as a **state graph** (LangGraph):

```
research → write → critique → (router)
                              ├─ accept → END
                              ├─ revise_write → write
                              └─ more_research → research
```

### Minimal Multi-Agent Simulation

In [ ]:
def researcher(topic: str, gaps: list = None) -> str:
    notes = f"Research notes on '{topic}': cyclic graphs, conditional routing, checkpointing."
    if gaps:
        notes += f" Additional coverage for: {gaps}"
    return notes

def writer(notes: str, feedback: str = "") -> str:
    draft = f"# Brief based on notes\n\n{notes}\n\n## Implications\n- Use in production workflows."
    if feedback:
        draft += f"\n\n(Revised: {feedback})"
    return draft

def critic(draft: str) -> dict:
    score = 7.5 if "Implications" in draft else 5.5
    return {"score": score, "feedback": "Add more concrete implications" if score < 8 else "Good"}

# Simple orchestration loop
def multi_agent_run(topic: str, max_rounds: int = 3):
    notes = researcher(topic)
    draft = writer(notes)
    
    for r in range(max_rounds):
        ev = critic(draft)
        print(f"Round {r+1}: score={ev['score']}")
        if ev["score"] >= 8:
            print("Accepted")
            return draft
        # Decide: more research or just revise writing
        if "concrete" in ev["feedback"].lower():
            notes = researcher(topic, gaps=["concrete examples"])
        draft = writer(notes, ev["feedback"])
    
    return draft

result = multi_agent_run("LangGraph evaluation loops")
print("\nFinal brief preview:")
print(result[:400])

## Module 5: Production Best Practices

- Always set **max iterations** and token/cost budgets
- Detect **stagnation** (same score for N rounds)
- Use **structured JSON** between agents
- Keep **full traces** for debugging
- Prefer **LangGraph** (or equivalent) for explicit cycles + checkpointing
- Separate generator and critic models when possible
- Return the **best version so far** if max attempts are hit
- Add human-in-the-loop for high-stakes decisions

### Next Steps for You
1. Replace the mock functions with real LLM calls (OpenAI, Anthropic, Hugging Face Inference, etc.)
2. Add a proper rubric and structured output parsing
3. Port the loop into LangGraph for production control flow
4. Add observability (LangSmith / Langfuse)
5. Experiment with multi-agent roles on a real task of yours

---
**Course complete.**  
You now have the full path: clear task → evaluation → bounded iterative loop → multi-agent orchestration.

Start small (Writer + Critic), measure improvement, then expand the graph.